In [1]:
import abc
import numpy as np
import pandas as pd
from numpy import linalg
from typing import List
import time

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error

# Реализация 

## Базовые Loss функии

In [2]:
class BaseLoss(abc.ABC):
    @abc.abstractmethod
    def calc_loss(self, X: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
        """
        Инструкция(чуть позже)
        """
        return NotImplementedError

    @abc.abstractmethod
    def calc_grad(self, X: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
        """
        Инструкция(чуть позже)
        """
        return NotImplementedError

### MSE - mean squared error

#### Classic


$Q(X, y, w) = \frac{1}{\ell} \lVert  Xw - y \rVert_2 ^ 2$

$\nabla_w Q(X, y, w) = \frac{2}{\ell} X^T(Xw - y)$

In [3]:
class MSELoss(BaseLoss):
    def calc_loss(self, X: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
        n = y.shape[0]
        L = X @ w - y
        
        Q = linalg.norm(L) ** 2 / n
        return Q

    def calc_grad(self, X: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
        n = y.shape[0]
        L = X @ w - y

        Grad = 2 * X.T @ L / n
        return Grad

#### L1 - Lasso


$Q(X, y, w) = \frac{1}{\ell} \lVert  Xw - y \rVert ^ 2 + \lambda \lVert w \rVert_1$

$\nabla_w Q(X, y, w) = \frac{2}{\ell} X^T(Xw - y) + \lambda \cdot \text{sgn}(\hat{w})$

In [4]:
class MSEL1Loss(BaseLoss):
    def __init__(self, coef: float = 1.):
        self.coef = coef

    def calc_loss(self, X: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
        n = y.shape[0]
        L = X @ w - y
        
        MSE = linalg.norm(L) ** 2 / n
        L1 = self.coef * np.sum(np.abs(w[1:]))
        Q = MSE + L1
        
        return Q

    def calc_grad(self, X: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
        n = y.shape[0]
        L = X @ w - y

        Grad_MSE = 2 * X.T @ L / n
        Grad_L1 = self.coef * np.sign(w)
        Grad_L1[0] = 0
        Grad = Grad_MSE + Grad_L1
        
        return Grad

#### L2 - Ridge

$Q(X, y, w) = \frac{1}{\ell} \lVert  Xw - y \rVert_2 ^ 2 + \lambda \lVert w \rVert_2^2$

$\nabla_w Q(X, y, w) = \frac{2}{\ell} X^T(Xw - y) + 2 \lambda w$

In [5]:
class MSEL2Loss(BaseLoss):
    def __init__(self, coef: float = 1.):
        self.coef = coef

    def calc_loss(self, X: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
        n = y.shape[0]
        L = X @ w - y
        
        MSE = linalg.norm(L) ** 2 / n
        L2 = self.coef * linalg.norm(w[1:]) ** 2
        Q = MSE + L2

        return Q

    def calc_grad(self, X: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
        n = y.shape[0]
        L = X @ w - y

        Grad_MSE = 2 * X.T @ L / n
        Grad_L2 = 2 * self.coef * w
        Grad_L2[0] = 0
        Grad = Grad_MSE + Grad_L2

        return Grad

## Градиентные спуски

### Classic Gradient Descent

$w^{(k)} = w^{(k-1)} - \eta \nabla_w Q(w^{(k-1)})$

In [44]:
def gradient_descent(w_init: np.ndarray, X: np.ndarray, y: np.ndarray,
                     loss: BaseLoss, lr: float = 0.1, decay: float=  0., power: float = 1.0, n_iterations: int = 100000,
                     tol: float = 1e-8) -> (List[np.ndarray], List[float]):
    W = [w_init]
    L = [loss.calc_loss(X, y, w_init)]
    
    for i in range(n_iterations):
        lr_i =  lr / (1 + decay * i) ** power
        w_init_new = w_init - lr_i * loss.calc_grad(X, y, w_init)
        
        W.append(w_init_new)
        L.append(loss.calc_loss(X, y, w_init_new))
        if linalg.norm(w_init - w_init_new) < tol:
            break
        w_init = w_init_new

    return W, L
    

### Stochactic Gradient Descent

Пусть $$i \sim U\{1, \dots, m\}$$ $$w^{(k)} = w^{(k-1)} - \eta \sum\limits_{j=1}^n \nabla_w q_j(w^{(k-1)}),$$ где $q_i$ - градиент на $i$ элементе

`stochactic_gradient_descent` если не подать параметр batch_size то он поумолчанию 1, ну а так будет mini_batch версия

In [45]:
def stochactic_gradient_descent(w_init: np.ndarray, X: np.ndarray, y: np.ndarray,
                                loss: BaseLoss, lr: float = 0.1, decay: float = 0.0, power: float = 1.0, batch_size: int = 1, n_iteration = 100000,
                                tol: float = 1e-8) -> (List[np.ndarray], List[float]):
    W = [w_init]
    L = [loss.calc_loss(X, y, w_init)]
    for i in range(n_iteration):
        lr_i =  lr / (1 + decay * i) ** power
        index = np.random.choice(X.shape[0], size=batch_size)
        X_batch = X[index]
        y_batch = y[index]
    
        w_init_new = w_init - lr_i * loss.calc_grad(X_batch, y_batch, w_init)
        W.append(w_init_new)
        L.append(loss.calc_loss(X_batch, y_batch, w_init_new))
        if linalg.norm(w_init - w_init_new) < tol:
            break
        w_init = w_init_new
        
    return W, L

## Линейная регрессия

In [47]:
class LinearRegressionCGD:
    def __init__(self, loss: BaseLoss, lr: float = 0.1, decay: float =  0.0,
                 power: float = 1.0, n_iterations: int = 10000, tol: float = 1e-8) -> None:
        self.loss = loss
        self.lr = lr
        self.decay = decay
        self.power = power
        self.n_iterations = n_iterations
        self.tol = tol
        
        self.w = None
        self.history_w = None
        self.history_l = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'LinearRegressionCGD':

        # add intercept term
        X = np.hstack([np.ones([X.shape[0], 1]), X])
        
        # initialization weight
        w_start = np.random.normal(size=X.shape[-1])

        self.history_w, self.history_l = gradient_descent(
            w_init = w_start,
            X = X,
            y = y,
            loss = self.loss,
            lr = self.lr,
            decay = self.decay,
            power = self.power,
            n_iterations = self.n_iterations,
            tol = self.tol
        )

        self.w = self.history_w[-1]

        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        assert self.w is not None, "Модель ещё не была обучена! Используйте model.fit()"
        
        X = np.hstack([np.ones([X.shape[0], 1]), X])
        y = np.dot(X, self.w)
        
        return y

In [52]:
class LinearRegressionSGD:
    def __init__(self, loss: BaseLoss, lr: float = 0.1, decay: float = 0.0, 
                 power: float = 1.0, batch_size: int = 1, n_iteration: int = 10000, tol: float = 1e-8) -> None:
        self.loss = loss
        self.lr = lr
        self.decay = decay
        self.power = power
        self.batch_size = batch_size
        self.n_iteration = n_iteration
        self.tol = tol

        self.w = None
        self.history_w = None
        self.history_l = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'LinearRegressionSGD':
        
        # add intercept term
        X = np.hstack([np.ones([X.shape[0], 1]), X])

        # initialization weight
        w_start = np.random.normal(size=X.shape[-1])

        self.history_w, self.history_l = stochactic_gradient_descent(
            w_init = w_start,
            X = X,
            y = y,
            loss = self.loss,
            lr = self.lr,
            decay = self.decay,
            power = self.power,
            batch_size = self.batch_size,
            n_iteration = self.n_iteration,
            tol = self.tol
        )
        self.w = self.history_w[-1]
        
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        assert self.w is not None, "Модель ещё не была обучена! Используйте model.fit()"

        X = np.hstack([np.ones([X.shape[0], 1]), X])
        y = np.dot(X, self.w)
        
        return y

# Создание синтетических данных

In [39]:
np.random.seed(1337)
n_features = 2
n_objects = 3000

w_true = np.random.normal(size=(n_features,))
X = np.random.uniform(-5, 5, (n_objects, n_features))
X *= (np.arange(n_features) * 2 + 1)[np.newaxis, :]
y = X.dot(w_true) + np.random.normal(0, 1, (n_objects))
w_init = np.random.uniform(-2, 2, (n_features))

print(X.shape)
print(y.shape)

(3000, 2)
(3000,)


In [40]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1337)

scaler = StandardScaler()
scaler.fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [55]:
lin = LinearRegression()
lin.fit(X_train, y_train)

print("MSE test: ", mean_squared_error(lin.predict(X_test), y_test))
print("MSE train: ", mean_squared_error(lin.predict(X_train), y_train))




print("MSE test: ", mean_squared_error(my_LinReg.predict(X_test), y_test))
print("MSE train: ", mean_squared_error(my_LinReg.predict(X_train), y_train))

MSE test:  1.0462839851866914
MSE train:  1.0206500708142756
MSE test:  1.5385858765541773
MSE train:  1.4984492256298765
